Procesado de datos de las baterías 13300 para el TFG de reciclado de baterías de vape.

Primero analizaremos los ciclos y veremos el SoH de las baterías

In [1]:
"""
IMPORTS & SETUP

This section imports all necessary modules from the src package.
"""

# System imports
from pathlib import Path

# Data processing
import numpy as np
import pandas as pd

# Project imports
from src.data_io.file_organization import organizar_archivos_por_fecha, organizar_todos_los_cicladores
from src.processing.cycling import trim_txt_files, build_soh_matrix
from src.analysis.soh import find_best_18
from src.data_io.export import export_best_18_batteries
from config import CYCLER_RANGE

print("OK: All modules imported successfully!")


OK: All modules imported successfully!


In [2]:
"""
STEP 1: ORGANIZE RAW FILES BY DATE

Organize raw cycler files into date-based subfolders for easier tracking.
This preserves the original raw data while creating a structured processed hierarchy.
"""

# Organize all cycler data (ciclador_1 through ciclador_6)
results = organizar_todos_los_cicladores()


Organizando archivos de todos los cicladores por fecha...

Procesando: data/raw/cicladores/ciclador_1 ...
  OK Organizados: 46
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_2 ...
  OK Organizados: 58
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_3 ...
  OK Organizados: 52
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_4 ...
  OK Organizados: 48
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_5 ...
  OK Organizados: 50
  SKIP Omitidos: 0

Procesando: data/raw/cicladores/ciclador_6 ...
  OK Organizados: 46
  SKIP Omitidos: 0

OK: Todas las carpetas han sido organizadas!

Resumen total:
  Organizados: 300
  Omitidos: 0


In [3]:
"""
STEP 2: TRIM CYCLING FILES

Process cycling files to keep only:
  - Header row (column names)
  - Last data row (final charge state)

This reduces file size and focuses on the final state of each battery after cycling.
"""

results = trim_txt_files()
print(f"\nTrimming complete!")

OK Trimmed : 129 .txt files
SKIP Skipped : 21 .txt files (already 2 lines or empty)
DELETE Deleted : 150 .dat files
ERROR Errors  : 0 files

Trimming complete!


Hemos limpiado el sistema de archivos: El resultado en la carpeta processed es una serie de archivos .txt en el que solamente se encuentra el último estado del ciclo que corresponde. Lo que nos interesa a continuación es casar cada archivo txt de Charge con la última carga correspondiente a cada batería.

In [4]:
"""
STEP 3: BUILD STATE OF HEALTH (SoH) MATRIX

Extract the final SoH value for each battery from its trimmed cycling file.
Creates a CSV matrix with battery numbers, SoH values, and impedance (to be filled in next).

Expected CSV columns: Numero, SoH(Ah), Impedancia
"""

baterias = build_soh_matrix()
print(f"\nOK: Built SoH matrix with {len(baterias)} batteries")



OK: SoH matrix saved to: '/home/elias/Documents/TFG/Procesado_Datos/TFG_Elias/data/processed/matriz_baterias.csv'

OK: Built SoH matrix with 36 batteries


In [5]:
"""
STEP 4: SELECT BEST COMPATIBLE BATTERIES

Find the 18 batteries with the most similar SoH values.
This ensures even load distribution and balanced performance in the final pack.

Algorithm: Finds a sliding window of 18 batteries (sorted by SoH) with minimum spread.
"""

best_baterias = [b for b in find_best_18(baterias) if b is not None]

# Print results
print(f"\nOK: Selected {len(best_baterias)} compatible batteries:")
print(f"\nBattery Numbers: {[b.numero for b in best_baterias]}")



OK: Selected 18 compatible batteries:

Battery Numbers: [1, 2, 6, 7, 9, 10, 12, 13, 15, 17, 20, 21, 24, 26, 29, 33, 35, 36]


Con esto, ya hemos encontrado las 18 baterías más compatibles para formar un sólo pack. Ahora, solo queda montarlo y ensayarlo.

In [6]:
"""
FINAL RESULT: DETAILED INFORMATION

Display detailed battery information for the selected pack.
"""

print("\n" + "="*60)
print("SELECTED BATTERY PACK - DETAILED INFORMATION")
print("="*60)

for bat in best_baterias:
    print(f"  {bat}")

print("\n" + "="*60)
print("OK: Analysis complete! Ready for testing.")
print("="*60)



SELECTED BATTERY PACK - DETAILED INFORMATION
  Batería #1 | SoH: 0.335735Ah | Impedancia: 0.0863535Ω
  Batería #2 | SoH: 0.340149Ah | Impedancia: 0.170307Ω
  Batería #6 | SoH: 0.345978Ah | Impedancia: 0.1528916Ω
  Batería #7 | SoH: 0.339422Ah | Impedancia: 0.1690303Ω
  Batería #9 | SoH: 0.342146Ah | Impedancia: 0.0899847Ω
  Batería #10 | SoH: 0.340374Ah | Impedancia: 0.136096Ω
  Batería #12 | SoH: 0.335763Ah | Impedancia: 0.1105963Ω
  Batería #13 | SoH: 0.343665Ah | Impedancia: 0.3294793Ω
  Batería #15 | SoH: 0.346292Ah | Impedancia: 0.0862554Ω
  Batería #17 | SoH: 0.333173Ah | Impedancia: 0.1158241Ω
  Batería #20 | SoH: 0.344466Ah | Impedancia: 0.201654Ω
  Batería #21 | SoH: 0.345323Ah | Impedancia: 0.0715328Ω
  Batería #24 | SoH: 0.334309Ah | Impedancia: 0.1195215Ω
  Batería #26 | SoH: 0.344176Ah | Impedancia: 0.2552712Ω
  Batería #29 | SoH: 0.337994Ah | Impedancia: 0.1004127Ω
  Batería #33 | SoH: 0.334904Ah | Impedancia: 0.1384966Ω
  Batería #35 | SoH: 0.340035Ah | Impedancia: 0.16

In [7]:
"""
EXPORT RESULTS: SAVE BEST BATTERIES TO CSV

Export the selected 18 batteries to a CSV file in data/output/ directory.
This file contains the battery numbers and their SoH and impedance values.
"""

output_filepath = export_best_18_batteries(best_baterias)
print(f"\nOK: Best 18 batteries exported to:")
print(f"    {output_filepath}")

# Display the exported data
exported_df = pd.read_csv(output_filepath)
print(f"\nExported data preview:")
print(exported_df)



OK: Best 18 batteries exported to:
    /home/elias/Documents/TFG/Procesado_Datos/TFG_Elias/data/output/mejores_18_baterias.csv

Exported data preview:
    numero       soh  impedancia
0        1  0.335735    0.086353
1        2  0.340149    0.170307
2        6  0.345978    0.152892
3        7  0.339422    0.169030
4        9  0.342146    0.089985
5       10  0.340374    0.136096
6       12  0.335763    0.110596
7       13  0.343665    0.329479
8       15  0.346292    0.086255
9       17  0.333173    0.115824
10      20  0.344466    0.201654
11      21  0.345323    0.071533
12      24  0.334309    0.119522
13      26  0.344176    0.255271
14      29  0.337994    0.100413
15      33  0.334904    0.138497
16      35  0.340035    0.162362
17      36  0.339868    0.113604
